# AlphaQuest Capital — BTC Macro-Sentiment Signal Pipeline

This notebook provides a step-by-step walkthrough of the AlphaQuest Bitcoin trading strategy. We follow a rigorous quant-finance workflow:

1.  **ETL**: Fetching Bitcoin prices, FRED Macro signals, and Crypto Sentiment.
2.  **Feature Engineering**: Computing technical indicators and composite macro scores.
3.  **Machine Learning**: Training an ensemble of models (LR, RF, GB, XGB, LGB, NN) to predict tomorrow's price direction.
4.  **Backtesting**: Simulating the strategy performance with a 5% stop-loss kill switch.

---

## 1. Imports and environment setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import requests
import json
import numpy  as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime, timedelta

# ML
from sklearn.linear_model       import LogisticRegression
from sklearn.ensemble           import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.preprocessing      import StandardScaler
from sklearn.metrics            import (precision_score, recall_score, f1_score,
                                        accuracy_score, roc_auc_score,
                                        classification_report, confusion_matrix,
                                        ConfusionMatrixDisplay)
from sklearn.calibration        import CalibratedClassifierCV
from sklearn.neural_network     import MLPClassifier

# Advanced ML
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

print("Environment initialized.")

## 2. Configuration & API Keys

We use the same configuration as the production script.

In [ ]:
FRED_API_KEY = os.environ.get("FRED_API_KEY", "")

if not FRED_API_KEY:
    try:
        import toml
        if os.path.exists(".streamlit/secrets.toml"):
            secrets = toml.load(".streamlit/secrets.toml")
            FRED_API_KEY = secrets.get("FRED_API_KEY", "")
    except:
        pass

START_DATE      = "2020-01-01"
END_DATE        = "2024-12-31"
SIGNAL_THRESHOLD = 0.00
PROBA_CUTOFF    = 0.50
OUTPUT_DIR      = "."

CORE_FEATURES = [
    "trend_score",
    "macro_pressure",
    "rsi",
    "fear_greed",
    "sp500_ret1",
    "adx",
    "gtrends_momentum",  # Optimized Search Interest
]

print(f"Range: {START_DATE} to {END_DATE}")
if not FRED_API_KEY:
    print("⚠️ FRED_API_KEY not found. Please set it in .streamlit/secrets.toml.")

## 3. ETL Phase: Data Acquisition

We pull data from Yahoo Finance, FRED, and Alternative.me.

In [ ]:
import yfinance as yf
from fredapi import Fred

def fetch_data():
    # BTC Prices
    print("[ETL] Fetching BTC...")
    btc = yf.download("BTC-USD", start=START_DATE, end=END_DATE, interval="1d", progress=False)
    btc.columns = [c.lower() for c in btc.columns]
    btc = btc[["open", "high", "low", "close", "volume"]].copy()
    btc.index = btc.index.tz_localize(None)

    # FRED Macro
    if FRED_API_KEY:
        print("[ETL] Fetching FRED...")
        fred = Fred(api_key=FRED_API_KEY)
        dff = fred.get_series("DFF", observation_start=START_DATE, observation_end=END_DATE)
        sp500 = fred.get_series("SP500", observation_start=START_DATE, observation_end=END_DATE)
        yield_10y = fred.get_series("DGS10", observation_start=START_DATE, observation_end=END_DATE)
        macro = pd.DataFrame({"fed_rate": dff, "sp500": sp500, "bond_yield": yield_10y}).ffill()
    else:
        macro = pd.DataFrame(index=btc.index)

    # Sentiment (Fear & Greed)
    print("[ETL] Fetching Fear & Greed...")
    fgi_res = requests.get("https://api.alternative.me/fng/?limit=0").json()["data"]
    fgi = pd.DataFrame(fgi_res)
    fgi["date"] = pd.to_datetime(fgi["timestamp"], unit="s")
    fgi.set_index("date", inplace=True)
    fgi["fear_greed"] = fgi["value"].astype(float)
    fgi = fgi[["fear_greed"]].sort_index().tz_localize(None)

    return btc, macro, fgi

btc_raw, macro_raw, fgi_raw = fetch_data()
print(f"BTC rows: {len(btc_raw)}, FRED rows: {len(macro_raw)}, FGI rows: {len(fgi_raw)}")
display(btc_raw.tail(3))

## 4. Feature Engineering

We calculate technical metrics and combine them into macro scores. 

**Note on Google Trends**: We use a `momentum` feature (3d/7d rolling average) to capture the 2-3 day delayed reaction in search volume that often follows market volatility.

In [ ]:
def build_features(btc, macro, fgi):
    df = btc.copy()
    
    # 1. Technical Indicators
    df['ma5'] = df['close'].rolling(5).mean()
    df['ma20'] = df['close'].rolling(20).mean()
    df['ma_ratio'] = df['ma5'] / df['ma20']
    
    ema12 = df['close'].ewm(span=12, adjust=False).mean()
    ema26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema12 - ema26
    df['macd_sig'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_hist'] = df['macd'] - df['macd_sig']
    
    delta = df['close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # ADX
    high, low, close = df['high'], df['low'], df['close']
    tr = pd.concat([high-low, (high-close.shift(1)).abs(), (low-close.shift(1)).abs()], axis=1).max(axis=1)
    atr14 = tr.ewm(span=14, adjust=False).mean()
    up = high.diff(); down = low.diff().mul(-1)
    pos = up.where((up > down) & (up > 0), 0.0).ewm(span=14).mean()
    neg = down.where((down > up) & (down > 0), 0.0).ewm(span=14).mean()
    df['adx'] = 100 * ( (pos - neg).abs() / (pos + neg).replace(0, np.nan) ).ewm(span=14).mean()

    # 2. Integration
    df = df.join(macro.reindex(df.index).ffill()).join(fgi.shift(1).reindex(df.index).ffill())
    
    # 3. Composite Scores
    df['trend_score'] = (df['close'] / df['ma20'] - 1) + (df['macd_hist'] / df['close'].replace(0, np.nan))
    df['macro_pressure'] = (df['fed_rate'].diff() + df['bond_yield'].diff()).clip(-0.5, 0.5)
    df['sp500_ret1'] = df['sp500'].pct_change()

    # 4. Search Momentum (Optimized for Lead-Lag Correlation)
    # We assume Search Data is available via 'gtrends' from cache/api in production script
    if 'gtrends' in df.columns:
        df['gtrends_sma3'] = df['gtrends'].rolling(3).mean()
        df['gtrends_sma7'] = df['gtrends'].rolling(7).mean()
        df['gtrends_momentum'] = df['gtrends_sma3'] / df['gtrends_sma7'].replace(0, np.nan)
    else:
        df['gtrends_momentum'] = 1.0
    
    # 5. Target
    df['target'] = (df['close'].shift(-1) > df['close'] * (1 + SIGNAL_THRESHOLD)).astype(int)
    
    return df.dropna()

df_proc = build_features(btc_raw, macro_raw, fgi_raw)
print(f"Feature matrix build: {df_proc.shape}")
display(df_proc[CORE_FEATURES + ['target']].head())

## 5. Model Training & Comparison

We train 6 different types of models to find the assembly consensus.

In [ ]:
# Split
train_df = df_proc[df_proc.index < '2023-01-01']
val_df   = df_proc[(df_proc.index >= '2023-01-01') & (df_proc.index < '2024-01-01')]
test_df  = df_proc[df_proc.index >= '2024-01-01']

X_train, y_train = train_df[CORE_FEATURES], train_df['target']
X_val, y_val     = val_df[CORE_FEATURES],   val_df['target']
X_test, y_test   = test_df[CORE_FEATURES],  test_df['target']

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

models = {}
print("[Training] Logistic Regression...")
lr = LogisticRegression(class_weight='balanced', random_state=42).fit(X_train_sc, y_train)
models['LogisticRegression'] = ('scaled', lr)

print("[Training] Random Forest...")
rf = RandomForestClassifier(n_estimators=1000, max_depth=3, class_weight='balanced', random_state=42).fit(X_train, y_train)
models['RandomForest'] = ('raw', rf)

if HAS_XGB:
    print("[Training] XGBoost...")
    xgb_m = xgb.XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.02, random_state=42).fit(X_train, y_train)
    models['XGBoost'] = ('raw', xgb_m)

print("[Training] Voting Ensemble...")
ens = VotingClassifier(estimators=[('lr', lr), ('rf', rf)], voting='soft').fit(X_train_sc, y_train)
models['EnsembleVoter'] = ('scaled', ens)

print(f"Finished training {len(models)} models.")

## 6. Evaluation Matrix

Which model is the most reliable?

In [ ]:
eval_results = []
for name, (type, model) in models.items():
    X_eval = X_test_sc if type == 'scaled' else X_test
    y_prob = model.predict_proba(X_eval)[:, 1]
    y_pred = (y_prob >= PROBA_CUTOFF).astype(int)
    
    eval_results.append({
        "Model": name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })
    
metrics_df = pd.DataFrame(eval_results)
display(metrics_df)

## 7. Strategy Backtest (Investment Committee View)

We run the strategy vs. Buy & Hold for the year 2024.

In [ ]:
def run_backtest(model, X, df):
    proba = model.predict_proba(X)[:, 1]
    signal = (proba >= PROBA_CUTOFF).astype(int)
    
    bt = df[['close']].copy()
    bt['ret'] = bt['close'].pct_change().shift(-1)
    bt['signal'] = signal
    
    # Fee & Kill Switch
    bt['strat_ret'] = bt['signal'].shift(1) * bt['ret'] - (bt['signal'].diff().abs() * 0.001)
    bt.loc[bt['strat_ret'].shift(1) < -0.05, 'strat_ret'] = 0  # Kill switch
    
    bt['cum_strat'] = (1 + bt['strat_ret'].fillna(0)).cumprod()
    bt['cum_bnh']   = (1 + bt['ret'].fillna(0)).cumprod()
    return bt

best_model_name = metrics_df.sort_values("F1", ascending=False).iloc[0]["Model"]
m_type, m_obj = models[best_model_name]
X_bt = X_test_sc if m_type == 'scaled' else X_test

bt_df = run_backtest(m_obj, X_bt, test_df)

plt.figure(figsize=(15, 6))
plt.plot(bt_df['cum_strat'], label=f"AlphaQuest ({best_model_name})", lw=2, color='#00bcd4')
plt.plot(bt_df['cum_bnh'], label="Buy & Hold BTC", ls='--', color='gray', alpha=0.6)
plt.title(f"Strategy Cumulative Returns — 2024 Test Set", fontsize=14)
plt.legend()
plt.show()

## 8. Tomorrow's Trading Signal

The final output: What should we do as of today's close?

In [ ]:
latest_row = df_proc[CORE_FEATURES].iloc[[-1]]
if m_type == 'scaled':
    latest_row = scaler.transform(latest_row)

prob = m_obj.predict_proba(latest_row)[0, 1]
signal = "🟢 BUY (Go Long)" if prob >= PROBA_CUTOFF else "⚪ HOLD CASH"

print("="*60)
print(f"DATE: {df_proc.index[-1].date()}")
print(f"MODEL CONFIDENCE: {prob:.1%}")
print(f"STRATEGY SIGNAL:  {signal}")
print("="*60)